5_Adult_(Census_Income)

¿Quién lo creó y cómo se obtuvieron los datos?
El dataset es atribuido a Ronny Kohavi y Barry Becker, y fue extraído de los datos del Census Bureau de los Estados Unidos de 1994, con el objetivo de predecir si un individuo gana más o menos de 50.000 dólares al año. La extracción fue realizada por Barry Becker directamente de la base de datos del Censo de 1994, aplicando los siguientes filtros de limpieza: personas mayores de 16 años, con ingreso ajustado bruto mayor a 100, peso final mayor a 1, y más de 0 horas trabajadas por semana. Fue publicado en el UCI Machine Learning Repository en 1996 por Kohavi y Becker, en el contexto de una investigación sobre clasificación con Naive Bayes.

¿De qué trata?
Contiene información socioeconómica y demográfica de individuos del censo norteamericano de 1994, y se usa para estudiar la desigualdad de ingresos y los factores que determinan si una persona supera o no el umbral de 50.000 dólares anuales. Es uno de los datasets de clasificación más usados históricamente en benchmarks de Machine Learning.

¿Qué contiene?
El dataset contiene 48.842 instancias y 14 características, de tipos categórico y entero. Uci Las variables son: age (edad), workclass (tipo de empleador: privado, gubernamental, autónomo, etc.), fnlwgt (peso final de la muestra, variable de diseño censal), education (nivel educativo), education-num (años de educación en numérico), marital-status (estado civil), occupation (ocupación), relationship (rol familiar), race (raza), sex (sexo), capital-gain (ganancia de capital), capital-loss (pérdida de capital), hours-per-week (horas trabajadas por semana) y native-country (país de origen). El dataset contiene valores faltantes marcados con el símbolo de interrogación (?), principalmente en las variables workclass, occupation y native-country. 

Objetivo del modelo
Clasificación binaria: predecir si una persona tiene ingresos superiores a 50.000 dólares anuales (>50K) o inferiores/iguales (<=50K). Es un dataset con ligero desbalance de clases (aproximadamente 75% gana ≤50K y 25% gana >50K), lo cual requiere atención al evaluarlo con métricas como F1-score, precisión y recall, más allá de la simple exactitud. Es ideal para comparar modelos como Regresión Logística, Árboles de Decisión, Random Forest, y para practicar el tratamiento de variables mixtas (categóricas y numéricas).

In [1]:
# ============================================================
# LIBRERÍAS GENERALES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

In [2]:
# ── PASO 1: CARGA ───────────────────────────────────────────
# El archivo no tiene cabecera, la ponemos nosotros
nombres = ['age','workclass','fnlwgt','education','education_num',
           'marital_status','occupation','relationship','race',
           'sex','capital_gain','capital_loss','hours_per_week',
           'native_country','income']

df_adult = pd.read_csv('Datasets/5_Adult_(Census_Income)/adult.data',
                        names=nombres,
                        sep=', ',
                        engine='python',
                        na_values='?')   # '?' pasa a ser NaN

print('Shape:', df_adult.shape)
print('\nNulos:')
print(df_adult.isnull().sum())
print('\nDistribución del target:')
print(df_adult['income'].value_counts())

Shape: (32561, 15)

Nulos:
age                  0
workclass         1836
fnlwgt               0
education            0
education_num        0
marital_status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital_gain         0
capital_loss         0
hours_per_week       0
native_country     583
income               0
dtype: int64

Distribución del target:
income
<=50K    24720
>50K      7841
Name: count, dtype: int64


In [3]:
# ── PASO 2: LIMPIEZA ────────────────────────────────────────
# Los nulos están en workclass, occupation, native_country
# Rellenar con el valor más frecuente (moda)
for col in df_adult.select_dtypes(include='object').columns:
    df_adult[col] = df_adult[col].fillna(df_adult[col].mode()[0])

print('Nulos restantes:', df_adult.isnull().sum().sum())

Nulos restantes: 0


In [4]:
# ── PASO 3: CONVERTIR TEXTO A NÚMEROS ───────────────────────
# Target: >50K = 1, <=50K = 0
df_adult['income'] = df_adult['income'].str.strip()
y_adult = (df_adult['income'] == '>50K').values.astype(float)

# Variables de texto → códigos numéricos
cols_texto = ['workclass','education','marital_status','occupation',
              'relationship','race','sex','native_country']
for col in cols_texto:
    df_adult[col] = pd.Categorical(df_adult[col]).codes

# fnlwgt es un peso de diseño muestral, no es una feature real
cols_features_adult = ['age','workclass','education','education_num',
                       'marital_status','occupation','relationship',
                       'race','sex','capital_gain','capital_loss',
                       'hours_per_week','native_country']

X_raw_adult = df_adult[cols_features_adult].values.astype(float)
m_adult = y_adult.size

print('X shape:', X_raw_adult.shape)
print('y shape:', y_adult.shape)
print('Gana >50K:', int(y_adult.sum()), '| Gana <=50K:', int((y_adult==0).sum()))

X shape: (32561, 13)
y shape: (32561,)
Gana >50K: 7841 | Gana <=50K: 24720


In [5]:
# ============================================================
# FUNCIÓN DE BALANCEO — oversampling con numpy
# ============================================================
def balancear(X, y):
    """
    Balancea un dataset desbalanceado usando OVERSAMPLING.
    
    ¿Qué hace?
    - Identifica cuántos ejemplos tiene cada clase
    - La clase con MÁS ejemplos queda igual
    - Las clases con MENOS ejemplos se repiten (con reemplazo)
      hasta tener la misma cantidad que la clase mayoritaria
    - Al final todas las clases tienen el mismo número de filas
    
    ¿Por qué oversampling y no undersampling?
    - Undersampling borra filas → perdemos información
    - Oversampling agrega filas → mantenemos toda la información original
    """
    clases = np.unique(y)
    n_max  = max(np.sum(y == c) for c in clases)   # tamaño de la clase más grande
    
    X_bal_list = []
    y_bal_list = []
    
    for c in clases:
        idx    = np.where(y == c)[0]               # índices de esta clase
        n_c    = len(idx)                           # cuántos ejemplos tiene
        
        if n_c < n_max:
            # repetir filas hasta alcanzar n_max
            extra  = n_max - n_c
            idx_extra = np.random.choice(idx, size=extra, replace=True)
            idx_final = np.concatenate([idx, idx_extra])
        else:
            idx_final = idx
        
        X_bal_list.append(X[idx_final])
        y_bal_list.append(y[idx_final])
    
    X_bal = np.concatenate(X_bal_list, axis=0)
    y_bal = np.concatenate(y_bal_list, axis=0)
    
    # Mezclar aleatoriamente para no dejar todas las clases juntas
    perm  = np.random.permutation(len(y_bal))
    return X_bal[perm], y_bal[perm]

def mostrar_balance(y, nombre, antes_despues='ANTES'):
    """Imprime cuántos ejemplos tiene cada clase."""
    clases, cuentas = np.unique(y, return_counts=True)
    print(f'  Balance {antes_despues} — {nombre}:')
    for c, n in zip(clases, cuentas):
        print(f'    Clase {int(c)}: {n} ({n/len(y)*100:.1f}%)')

np.random.seed(42)   # para reproducibilidad
print('Funciones de balanceo definidas')

Funciones de balanceo definidas


In [6]:
# ── BALANCEO ─────────────────────────────────
mostrar_balance(y_adult, 'Adult Census', 'ANTES')

  Balance ANTES — Adult Census:
    Clase 0: 24720 (75.9%)
    Clase 1: 7841 (24.1%)


In [7]:
def featureNormalize(X):
    """
    Normaliza las features de X.
    Para cada columna: resta la media y divide por la desviación estándar.
    
    Retorna:
      X_norm : X normalizado (mismo tamaño que X)
      mu     : media de cada columna (se guarda para normalizar datos nuevos)
      sigma  : desviación estándar de cada columna
    """
    X_norm = X.copy()
    mu     = np.mean(X, axis=0)   # media de cada columna
    sigma  = np.std(X, axis=0)    # desviación estándar de cada columna
    X_norm = (X - mu) / sigma     # estandarización Z-score
    return X_norm, mu, sigma

In [9]:
# ── PASO 4: NORMALIZAR y COLUMNA DE UNOS ────────────────────
X_norm_adult, mu_adult, sigma_adult = featureNormalize(X_raw_adult)
X_bal_ad, y_bal_ad = balancear(X_norm_adult, y_adult)

mostrar_balance(y_bal_ad, 'Adult Census', 'DESPUÉS')

X_adult = np.concatenate([np.ones((m_adult, 1)), X_norm_adult], axis=1)

print('X_adult final:', X_adult.shape)
print('X:', X_adult.shape, '| y:', y_adult.shape)

  Balance DESPUÉS — Adult Census:
    Clase 0: 24720 (50.0%)
    Clase 1: 24720 (50.0%)
X_adult final: (32561, 14)
X: (32561, 14) | y: (32561,)
